# Notebook 3 — Interactive map of a frozen Planet pipeline

This notebook loads the model selected by a completed Planet training
notebook from `experiments/<name>/metrics/final_selection.json`: the frozen
weights and the threshold fitted on validation data. It does **not** choose
a model or threshold, and it must not be used to tune either.

Three training notebooks in this folder write that file, in three different
shapes, and this notebook auto-detects which one produced the experiment
you point it at:

| notebook | model | `final_selection.json` shape |
|---|---|---|
| `2_training_evaluation.ipynb` | scratch `base_cnn` / `siamese_cnn`, optional XGBoost spatial stacker | `run` / `variant` / `checkpoint` (/ `stacker`) |
| `gaza_damage_rsp_resnet50_spatial.ipynb` | MillionAID-pretrained RSP ResNet-50 Siamese (PyTorch) | `saved_model` ending in `.pt` |
| `gaza_damage_pretrained_spatial.ipynb` | ImageNet-pretrained EfficientNetB0 Siamese (Keras) | `saved_model` ending in `.keras` |

All three fit their threshold the same way — maximizing F1 on a validation
split — so it is always safe to compare `prob >= threshold` across families.

Unlike the Sentinel-1 pipeline, this dataset covers one fixed area (Gaza)
and one fixed pre/post assessment pair (May → July 2024) drawn from a
single already-downloaded PlanetScope GeoTIFF, so there is no city/date to
pick — only which completed experiment to visualize. Predictions are
written as one GeoJSON covering every footprint in the frozen dataset. The
map's satellite basemap is for orientation only; it is not necessarily
date-matched model evidence.

## 1. Setup — select a completed training experiment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
PROJECT_DIR = '/content/drive/MyDrive/War-Damage-Detection/Planet Data (Nils)'
os.environ['PLANET_DAMAGE_BASE'] = PROJECT_DIR
os.environ['PLANET_EXPERIMENTS_DIR'] = os.path.join(PROJECT_DIR, 'experiments')
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import json
import numpy as np
import geopandas as gpd
import torch

import pipeline
import importlib
importlib.reload(pipeline)
from pipeline import (
    CITY, MAY_DATE, PLANET_CHANNELS, EXPERIMENTS_DIR,
    load_dataset, PlanetPairDataset, build_planet_model, predict_planet_probs,
    neighbour_features, planet_band_indices, planet_raster_path,
    footprint_latitude_reference, latitude_quantile_spatial_split,
    experiment_dirs,
)

# Any completed experiment folder from any of the three training notebooks.
EXPERIMENT_NAME = 'planet_exp001_base_cnn_spatial'

print('experiments on Drive:',
      sorted(os.listdir(EXPERIMENTS_DIR)) if EXPERIMENTS_DIR.exists() else [])
OUT = experiment_dirs(EXPERIMENT_NAME)


def _load_json(path):
    if not path.exists():
        raise FileNotFoundError(
            f'{path} is missing. Finish training/validation for this '
            'experiment before using this notebook.')
    with path.open(encoding='utf-8') as fh:
        return json.load(fh)


TRAIN_CONFIG = _load_json(OUT['root'] / 'config.json')
NORMALIZATION = _load_json(OUT['metrics'] / 'normalization.json')
FINAL_SELECTION = _load_json(OUT['metrics'] / 'final_selection.json')

BANDS = list(NORMALIZATION['bands'])
LO = np.asarray(NORMALIZATION['lo'], dtype=np.float32)
HI = np.asarray(NORMALIZATION['hi'], dtype=np.float32)
FINAL_THRESHOLD = float(FINAL_SELECTION['threshold'])

# The three training notebooks write final_selection.json in three shapes;
# detect which one produced this experiment rather than assuming scratch.
saved_model = str(FINAL_SELECTION.get('saved_model', ''))
if 'run' in FINAL_SELECTION and 'variant' in FINAL_SELECTION:
    FAMILY = 'scratch'
elif saved_model.endswith('.pt'):
    FAMILY = 'rsp_resnet50'
elif saved_model.endswith('.keras'):
    FAMILY = 'efficientnet_b0'
else:
    raise RuntimeError(
        f'Unrecognized final_selection.json shape: {sorted(FINAL_SELECTION)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'experiment: {OUT["root"].name}  (family: {FAMILY})')
print(f'bands: {BANDS}')
print(f'validation-fitted threshold: {FINAL_THRESHOLD:.4f}')
print(json.dumps(FINAL_SELECTION, indent=2))

## 2. Score every footprint with the selected final pipeline

In [ ]:
data = load_dataset(load_images=True)
X, y = data['X'], data['y']
table = data['gdf'].reset_index(drop=True)
if not np.array_equal(data['system_index'], table['system:index'].astype(str).to_numpy()):
    raise RuntimeError('NPZ and parquet footprint IDs are not row-aligned')

# Whole-dataset inference intentionally lets spatial neighbour features cross
# split boundaries and scores buildings from every role. This is suitable
# for deployment/map display only. The training notebooks keep neighbourhood
# features inside each split and their sealed test numbers stay in
# final_test_metrics.json — this cell does not reopen model selection.
cnn_scores = None
if FAMILY == 'scratch':
    import joblib
    from torch.utils.data import DataLoader

    checkpoint = torch.load(
        FINAL_SELECTION['checkpoint'], map_location=device, weights_only=False)
    if not checkpoint.get('complete'):
        raise RuntimeError(
            f"{FINAL_SELECTION['checkpoint']} is not a completed model checkpoint")
    model = build_planet_model(
        checkpoint['model_name'], n_channels=2 * len(BANDS),
        **checkpoint['model_params']).to(device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()

    full_loader = DataLoader(
        PlanetPairDataset(X, y, np.arange(len(y)), bands=BANDS, lo=LO, hi=HI),
        batch_size=TRAIN_CONFIG['training']['predict_batch_size'],
        shuffle=False, num_workers=0)
    cnn_scores, _ = predict_planet_probs(model, full_loader, device)

    STACKER = None
    if FINAL_SELECTION['variant'] != 'cnn':
        if not FINAL_SELECTION.get('stacker'):
            raise RuntimeError(
                f"selected variant {FINAL_SELECTION['variant']} has no stacker path")
        STACKER = joblib.load(FINAL_SELECTION['stacker'])

    if STACKER is None:
        final_scores = cnn_scores
    else:
        xy = table[['centroid_x', 'centroid_y']].to_numpy(np.float64)
        features = neighbour_features(xy, cnn_scores, ks=TRAIN_CONFIG['stacking']['ks'])
        final_scores = STACKER.predict_proba(features)[:, 1]

elif FAMILY == 'rsp_resnet50':
    import torch.nn as nn
    import torch.nn.functional as tf_functional
    import torchvision
    from torch.utils.data import DataLoader

    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    FEATURE_DIM = 2048

    class RspSiameseDamage(nn.Module):
        '''Mirrors gaza_damage_rsp_resnet50_spatial.ipynb's model exactly.'''

        def __init__(self, backbone, resize_to, bands):
            super().__init__()
            if list(bands) != ['R', 'G', 'B']:
                raise ValueError('RSP ResNet-50 expects bands in RGB order')
            self.backbone = backbone
            self.resize_to = int(resize_to)
            self.half_channels = len(bands)
            self.register_buffer(
                'mean', torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
            self.register_buffer(
                'std', torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
            self.head = nn.Sequential(
                nn.Dropout(.4), nn.Linear(3 * FEATURE_DIM, 256),
                nn.ReLU(inplace=True), nn.Dropout(.3), nn.Linear(256, 1))

        def encode(self, image):
            image = tf_functional.interpolate(
                image, size=(self.resize_to, self.resize_to),
                mode='bilinear', align_corners=False)
            image = (image - self.mean) / self.std
            return self.backbone(image)

        def forward(self, images):
            pre = images[:, :self.half_channels]
            post = images[:, self.half_channels:]
            features = self.encode(torch.cat([pre, post], dim=0))
            pre_features, post_features = features.chunk(2, dim=0)
            joined = torch.cat(
                [pre_features, post_features,
                 torch.abs(post_features - pre_features)], dim=1)
            return self.head(joined).squeeze(1)

    saved = torch.load(
        FINAL_SELECTION['saved_model'], map_location=device, weights_only=False)
    backbone = torchvision.models.resnet50(weights=None)
    backbone.fc = nn.Identity()
    model = RspSiameseDamage(backbone, TRAIN_CONFIG['resize_to'], BANDS).to(device)
    model.load_state_dict(saved['model_state'])
    model.eval()

    full_loader = DataLoader(
        PlanetPairDataset(X, y, np.arange(len(y)), bands=BANDS, lo=LO, hi=HI),
        batch_size=TRAIN_CONFIG['batch_size'], shuffle=False, num_workers=0)
    final_scores, _ = predict_planet_probs(model, full_loader, device)

elif FAMILY == 'efficientnet_b0':
    import tensorflow as tf

    keras_model = tf.keras.models.load_model(FINAL_SELECTION['saved_model'])
    pre_channels, post_channels = planet_band_indices(BANDS)
    scale = np.maximum(HI - LO, 1e-6)

    def score_batches(rows, batch_size=512):
        for start in range(0, len(rows), batch_size):
            chunk = rows[start:start + batch_size]
            patches = np.asarray(X[chunk], dtype=np.float32)
            pre = np.transpose(patches[:, list(pre_channels)], (0, 2, 3, 1))
            post = np.transpose(patches[:, list(post_channels)], (0, 2, 3, 1))
            pre = np.clip((pre - LO) / scale, 0.0, 1.0)
            post = np.clip((post - LO) / scale, 0.0, 1.0)
            yield keras_model.predict([pre, post], verbose=0).ravel()

    final_scores = np.concatenate(list(score_batches(np.arange(len(y)))))

else:
    raise RuntimeError(f'Unhandled experiment family: {FAMILY}')

split_bands = {name: tuple(values) for name, values in TRAIN_CONFIG['split'].items()}
split = latitude_quantile_spatial_split(
    table, bands=split_bands, reference=footprint_latitude_reference())
role_by_row = np.full(len(table), 'unused', dtype='U6')
for role in ('train', 'stack', 'val', 'test'):
    role_by_row[split[role]] = role

predictions = table[['geometry']].copy()
predictions['prob'] = final_scores.astype(float)
if cnn_scores is not None:
    predictions['cnn_prob'] = cnn_scores.astype(float)
predictions['pred'] = (final_scores >= FINAL_THRESHOLD).astype(int)
predictions['class'] = y.astype(int)
for column in ('damage_pts', 'severity', 'confidence', 'max_change'):
    source_column = f'{column}_{MAY_DATE}'
    if source_column in table:
        predictions[column] = table[source_column]
predictions['split'] = role_by_row
predictions['city'], predictions['date'] = CITY, MAY_DATE
predictions['pipeline'] = f'{OUT["root"].name} ({FAMILY})'
predictions['threshold'] = FINAL_THRESHOLD

prediction_path = os.path.join(
    OUT['predictions'], f"{CITY}_{MAY_DATE}_{OUT['root'].name}_map.geojson")
predictions.to_file(prediction_path, driver='GeoJSON')
print(f'wrote {len(predictions):,} buildings to {prediction_path}')
predictions.head(3)

## 3. Interactive vector map

The map is a diagnostic display, not another evaluation step. Do not use
map errors to alter the chosen model, features, or threshold. For a new
experiment, make those choices on validation data in whichever training
notebook produced it.

In [ ]:
import folium
import matplotlib


def prob_to_color(p):
    return matplotlib.colors.to_hex(matplotlib.colormaps['RdYlGn_r'](float(p)))


def build_comparison_map(pred_gdf, max_features=10_000):
    '''Toggleable truth, confidence, and FP/FN layers on a map.'''
    g = pred_gdf.copy()
    if max_features is not None and len(g) > max_features:
        g = g.sample(max_features, random_state=0)
        print(f'showing a reproducible {len(g):,}-building sample')
    g['color_pred'] = [prob_to_color(p) for p in g['prob']]
    g['color_true'] = np.where(g['class'] == 1, '#C0392B', '#4C72B0')
    g['error_type'] = np.select(
        [(g['pred'] == 1) & (g['class'] == 0),
         (g['pred'] == 0) & (g['class'] == 1)],
        ['false positive', 'false negative'], default='correct')

    field_aliases = {
        'prob': 'damage probability', 'cnn_prob': 'raw CNN probability',
        'class': 'UNOSAT label (May)', 'pred': 'prediction',
        'error_type': 'outcome', 'split': 'latitude-quantile split role',
        'max_change': 'PWTT max_change',
    }
    fields = [c for c in field_aliases if c in g.columns]
    aliases = [field_aliases[c] for c in fields]

    center = [g.geometry.centroid.y.mean(), g.geometry.centroid.x.mean()]
    m = folium.Map(location=center, zoom_start=14, tiles=None)
    folium.TileLayer(
        tiles=('https://server.arcgisonline.com/ArcGIS/rest/services/'
               'World_Imagery/MapServer/tile/{z}/{y}/{x}'),
        attr='Esri World Imagery (undated; orientation only)',
        name='satellite basemap').add_to(m)

    def add_layer(frame, color_col, name, show, weight=1, fill=0.7, outline=None):
        folium.GeoJson(
            frame[['geometry'] + fields + [color_col]].to_json(), name=name, show=show,
            style_function=lambda f, cc=color_col, w=weight, fo=fill, ol=outline: {
                'color': ol or f['properties'][cc], 'fillColor': f['properties'][cc],
                'weight': w, 'fillOpacity': fo},
            tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases),
        ).add_to(m)

    add_layer(g, 'color_true', 'UNOSAT ground truth (May)', show=False)
    add_layer(g, 'color_pred', 'selected-pipeline confidence', show=True)
    wrong = g[g['error_type'] != 'correct']
    if len(wrong):
        add_layer(wrong, 'color_pred', 'false positives / negatives', show=True,
                  weight=3, fill=0.0, outline='#00FFFF')
    folium.LayerControl(collapsed=False).add_to(m)
    return m


build_comparison_map(predictions)

## 4. Lonboard map — fast vectors with building-level attributes

Lonboard renders the footprints on the GPU and is preferable to Folium for
the full Gaza footprint set. Cyan outlines mark false positives and false
negatives.

In [ ]:
!pip install -q lonboard

In [ ]:
import matplotlib as mpl
from lonboard import Map as LonboardMap, PolygonLayer
from lonboard.colormap import apply_continuous_cmap

g = predictions.to_crs(4326).copy()
fill = apply_continuous_cmap(
    g['prob'].to_numpy(), mpl.colormaps['RdYlGn_r'], alpha=0.7)
line = np.where(
    (g['pred'] != g['class']).to_numpy()[:, None],
    [0, 255, 255], [0, 0, 0]).astype('uint8')

lonboard_columns = [c for c in
    ['geometry', 'prob', 'cnn_prob', 'class', 'pred', 'split', 'max_change']
    if c in g.columns]
lonboard_layer = PolygonLayer.from_geopandas(
    g[lonboard_columns],
    get_fill_color=fill,
    get_line_color=line,
    line_width_min_pixels=0.5,
)
LonboardMap(lonboard_layer)

## 5. Local raster overlay — prediction probability and error

This rasterizes the predictions, which is much lighter than drawing every
footprint. Unlike the Sentinel-1 pipeline, PlanetScope imagery for Gaza is
one static local GeoTIFF that Notebook 1 already downloaded — there is no
Earth Engine composite to rebuild and no pinned orbit to look up.

In [ ]:
!pip install -q ipyleaflet

In [ ]:
import base64
import io
import matplotlib as mpl
import rasterio
from PIL import Image
from ipyleaflet import Map as LeafletMap, ImageOverlay, LayersControl, basemaps
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import Resampling, calculate_default_transform, reproject


def rasterize_predictions(frame, out_path, value='prob', scale_m=3):
    '''Paint one per-building value onto a compact local GeoTIFF.'''
    g = frame.to_crs(frame.estimate_utm_crs()).reset_index(drop=True)
    minx, miny, maxx, maxy = g.total_bounds
    width = int(np.ceil((maxx - minx) / scale_m)) + 1
    height = int(np.ceil((maxy - miny) / scale_m)) + 1
    transform = from_origin(minx, maxy, scale_m, scale_m)

    values = g[value].to_numpy(float)
    # Burn large polygons first so small buildings are not hidden by them.
    order = np.argsort(-g.geometry.area.to_numpy())
    shapes = [(g.geometry.iloc[i], float(values[i])) for i in order
              if np.isfinite(values[i])]
    array = rasterize(shapes, out_shape=(height, width), transform=transform,
                      fill=np.nan, all_touched=True, dtype='float32')
    with rasterio.open(
            out_path, 'w', driver='GTiff', height=height, width=width,
            count=1, dtype='float32', crs=g.crs, transform=transform,
            nodata=np.nan, tiled=True, blockxsize=256, blockysize=256,
            compress='deflate') as dst:
        dst.write(array, 1)
    print(f'{len(g):,} buildings -> {width}x{height} raster: {out_path}')
    return out_path


def add_raster_overlay(map_widget, tif, name, cmap='RdYlGn_r',
                       vmin=0, vmax=1, opacity=0.75, max_px=4000):
    '''Embed a local raster as one transparent PNG layer.'''
    with rasterio.open(tif) as src:
        transform, width, height = calculate_default_transform(
            src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
        if max(width, height) > max_px:
            factor = max_px / max(width, height)
            transform, width, height = calculate_default_transform(
                src.crs, 'EPSG:4326', src.width, src.height, *src.bounds,
                dst_width=max(1, int(width * factor)),
                dst_height=max(1, int(height * factor)))
        array = np.full((height, width), np.nan, np.float32)
        reproject(
            rasterio.band(src, 1), array, src_transform=src.transform,
            src_crs=src.crs, dst_transform=transform, dst_crs='EPSG:4326',
            resampling=Resampling.nearest, dst_nodata=np.nan)

    west, north = transform * (0, 0)
    east, south = transform * (width, height)
    normalized = np.clip((array - vmin) / (vmax - vmin), 0, 1)
    rgba = (mpl.colormaps[cmap](np.nan_to_num(normalized)) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(np.isfinite(array), int(opacity * 255), 0)
    buffer = io.BytesIO()
    Image.fromarray(rgba, mode='RGBA').save(buffer, format='PNG', optimize=True)
    url = 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()
    print(f'{name}: {len(buffer.getvalue()) / 1e6:.1f} MB PNG')
    map_widget.add_layer(ImageOverlay(url=url, bounds=((south, west), (north, east)), name=name))


raster_frame = predictions.copy()
# NaN keeps correct buildings transparent in the disagreement overlay.
raster_frame['error'] = np.where(
    raster_frame['pred'] != raster_frame['class'], 1.0, np.nan)
prediction_stem = os.path.splitext(os.path.basename(prediction_path))[0]
prob_tif = rasterize_predictions(
    raster_frame, os.path.join(OUT['predictions'], prediction_stem + '_prob.tif'), 'prob')
error_tif = rasterize_predictions(
    raster_frame, os.path.join(OUT['predictions'], prediction_stem + '_error.tif'), 'error')

raster_center = [predictions.geometry.centroid.y.mean(), predictions.geometry.centroid.x.mean()]
RASTER_MAP = LeafletMap(basemap=basemaps.Esri.WorldImagery, center=raster_center, zoom=14)
add_raster_overlay(RASTER_MAP, prob_tif, 'selected-pipeline probability')
add_raster_overlay(RASTER_MAP, error_tif, 'FP/FN', cmap='cool_r', opacity=0.95)
RASTER_MAP.add_control(LayersControl(position='topright'))
RASTER_MAP

## 6. Pre/post PlanetScope imagery — before/after slider

Choose `true colour` or `infrared`, then drag the divider. The stretch is
the same 2nd/98th-percentile scaling shared across the pre/post pair used
in Notebook 1's EDA.

In [ ]:
import base64
import io
import numpy as np
import rasterio
from PIL import Image
from ipyleaflet import Map as LeafletMap, ImageOverlay, SplitMapControl
from rasterio.warp import Resampling, transform_bounds

from pipeline import PLANET_CHANNELS, planet_raster_path


def read_view_arrays(view, max_px=1600):
    '''Decimated, percentile-stretched pre/post RGB arrays plus EPSG:4326 bounds.'''
    band_map = {'true colour': ('R', 'G', 'B'), 'infrared': ('NIR', 'R', 'G')}
    names = band_map[view]
    pre_idx = [PLANET_CHANNELS.index(f'ps_pre_{b}') + 1 for b in names]
    post_idx = [PLANET_CHANNELS.index(f'ps_post_{b}') + 1 for b in names]
    with rasterio.open(planet_raster_path()) as src:
        scale = min(1.0, max_px / max(src.width, src.height))
        out_h, out_w = max(1, round(src.height * scale)), max(1, round(src.width * scale))
        pre = src.read(pre_idx, out_shape=(3, out_h, out_w), resampling=Resampling.average)
        post = src.read(post_idx, out_shape=(3, out_h, out_w), resampling=Resampling.average)
        bounds4326 = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
    pre, post = pre.astype(np.float32), post.astype(np.float32)

    # Percentile scaling is shared across pre/post, matching Notebook 1's EDA.
    both = np.concatenate([pre.reshape(3, -1), post.reshape(3, -1)], axis=1)
    lo = np.percentile(both, 2, axis=1)
    hi = np.percentile(both, 98, axis=1)
    scale_ = np.maximum(hi - lo, 1e-6)

    def stretch(a):
        return np.clip((a - lo[:, None, None]) / scale_[:, None, None], 0, 1)

    pre_rgb = np.moveaxis(stretch(pre), 0, -1)
    post_rgb = np.moveaxis(stretch(post), 0, -1)
    return pre_rgb, post_rgb, bounds4326


def to_png_data_uri(rgb):
    rgba = np.dstack([rgb, np.ones(rgb.shape[:2], rgb.dtype)])
    buffer = io.BytesIO()
    Image.fromarray((rgba * 255).astype(np.uint8), mode='RGBA').save(
        buffer, format='PNG', optimize=True)
    return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()


VIEW = 'true colour'  # 'true colour' or 'infrared'
pre_rgb, post_rgb, (west, south, east, north) = read_view_arrays(VIEW)
overlay_bounds = ((south, west), (north, east))

split_map = LeafletMap(center=[(south + north) / 2, (west + east) / 2], zoom=13)
left_layer = ImageOverlay(url=to_png_data_uri(pre_rgb), bounds=overlay_bounds, name=f'pre-war {VIEW}')
right_layer = ImageOverlay(url=to_png_data_uri(post_rgb), bounds=overlay_bounds, name=f'post-event {VIEW}')
split_map.add_control(SplitMapControl(left_layer=left_layer, right_layer=right_layer))
split_map